# LSST Science Pipeline — Overview

The LSST Science Pipelines ("the Stack") process raw telescope images into
science-ready catalogs of galaxy shapes, fluxes, and positions. This notebook
series walks through every stage.

**Primary reference:** Bosch et al. (2018), [arXiv:1705.06766](https://arxiv.org/abs/1705.06766)

---

## The Full Processing Chain

```
┌─────────────────────────────────────────────────────────────────────┐
│                     SINGLE-FRAME PROCESSING                        │
│                                                                     │
│  Step 1: Instrument Signature Removal (ISR)                        │
│    Raw CCD → bias, dark, flat, crosstalk, brighter-fatter, ...     │
│                          │                                          │
│  Step 2: Characterization                                          │
│    Background estimation → star selection → PSF modeling            │
│                          │                                          │
│  Step 3: Calibration                                               │
│    Astrometry (WCS) + Photometry (zero-point) vs reference catalog │
│                          │                                          │
│  Step 4: Single-frame measurement                                  │
│    Detect sources → measure centroids, fluxes, shapes              │
│                                                                     │
└──────────────────────────┬──────────────────────────────────────────┘
                           │ calibrated exposures
┌──────────────────────────▼──────────────────────────────────────────┐
│                     COADD PROCESSING                                │
│                                                                     │
│  Step 5: Coaddition                                                │
│    Warp exposures to common grid (tracts/patches) → stack          │
│                          │                                          │
│  Step 6: Detection + Deblending                                    │
│    Source detection on coadd → scarlet multiband deblending         │
│                          │                                          │
│  Step 7: Measurement                                               │
│    Shapes (HSM), fluxes (CModel), centroids on coadds             │
│                          │                                          │
│  Step 8: Forced Photometry                                         │
│    Measure at fixed positions across all single-frame visits        │
│                                                                     │
└──────────────────────────┬──────────────────────────────────────────┘
                           │ catalogs
┌──────────────────────────▼──────────────────────────────────────────┐
│                     DOWNSTREAM SCIENCE                              │
│                                                                     │
│  Step 9: Photometric Redshifts (RAIL)                              │
│    Multiband fluxes → photo-z PDFs                                 │
│                                                                     │
│  Step 10: Shear Catalog                                            │
│    Quality cuts → shear calibration → science-ready catalog        │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

## The Butler — Data Access Layer

Everything in the LSST pipelines flows through the **Butler** (`lsst.daf.butler`).
The Butler is an abstraction layer that manages all data — you never open FITS
files directly. Instead you ask for datasets by type and data coordinates.

### Key concepts

| Concept | Description | Example |
|---------|-------------|--------|
| **Data repository** | A directory tree managed by Butler | `/repo/main` |
| **Dataset type** | What kind of data | `'calexp'`, `'deepCoadd'`, `'src'` |
| **Data ID** | Coordinates that identify a specific dataset | `{'visit': 903334, 'detector': 16}` |
| **Collection** | A named group of datasets (like a git branch) | `'HSC/runs/RC2/w_2024_38'` |
| **Tract / Patch** | Sky tiling for coadds (HEALPix-like) | tract=9813, patch=42 |

### Basic usage pattern

```python
from lsst.daf.butler import Butler

# Connect to a data repository
butler = Butler('/repo/main', collections=['HSC/runs/RC2/w_2024_38'])

# Retrieve a calibrated exposure
calexp = butler.get('calexp', visit=903334, detector=16)

# Retrieve a coadd
coadd = butler.get('deepCoadd', band='i', tract=9813, patch=42)

# Retrieve a source catalog
sources = butler.get('src', visit=903334, detector=16)
```

### Common dataset types

| Dataset type | Stage | Description |
|-------------|-------|-------------|
| `raw` | Input | Raw CCD exposure |
| `postISRCCD` | After ISR | Corrected single-frame image |
| `calexp` | After calibration | Fully calibrated single exposure |
| `src` | Single-frame | Source catalog from one visit |
| `deepCoadd` | Coaddition | Stacked image |
| `deepCoadd_meas` | Coadd measurement | Source catalog from coadd |
| `objectTable` | Final | Merged multi-band object catalog |

## Data Model: Exposures, Images, and Catalogs

### Exposure (`lsst.afw.image.ExposureF`)
The central data object. An Exposure bundles together:

- **image** — the pixel values (flux in counts or nJy)
- **variance** — per-pixel noise variance
- **mask** — bitplane mask (bad pixels, cosmic rays, saturated, etc.)
- **PSF** — the point spread function model at any position
- **WCS** — world coordinate system (pixel → sky mapping)
- **PhotoCalib** — photometric calibration (counts → physical flux)
- **metadata** — FITS headers, filter info, etc.

```python
# Accessing exposure components
image = calexp.image.array         # numpy array of pixel values
variance = calexp.variance.array   # noise variance
mask = calexp.mask.array           # bitmask
psf = calexp.getPsf()              # PSF model
wcs = calexp.getWcs()              # WCS
photocal = calexp.getPhotoCalib()  # flux calibration

# Evaluate PSF at a point
import lsst.geom as geom
psf_image = psf.computeImage(geom.Point2D(500, 600))
```

### Source Table (`lsst.afw.table.SourceCatalog`)
An in-memory table of detected/measured sources with columns for every
measurement plugin that ran:

```python
sources = butler.get('src', dataId)
print(sources.schema.getNames())  # all column names

# Access columns
ra = sources['coord_ra']         # radians
dec = sources['coord_dec']
psf_flux = sources['base_PsfFlux_instFlux']
e1 = sources['ext_shapeHSM_HsmShapeRegauss_e1']
e2 = sources['ext_shapeHSM_HsmShapeRegauss_e2']
```

## Notebook Guide

The remaining notebooks in this directory walk through each step:

| Notebook | Step | Description |
|----------|------|-------------|
| `02_isr.ipynb` | 1 | Instrument Signature Removal |
| `03_characterization.ipynb` | 2 | Background estimation + PSF modeling |
| `04_calibration.ipynb` | 3 | Astrometric + photometric calibration |
| `05_coaddition.ipynb` | 4-5 | Warping + stacking |
| `06_detection_deblending.ipynb` | 6 | Source detection + scarlet deblending |
| `07_measurement.ipynb` | 7-8 | Shape + flux measurement (HSM, CModel) |
| `08_photoz.ipynb` | 9 | Photometric redshifts (RAIL) |

Each notebook explains the **physics**, the **algorithm**, the **LSST task** that
implements it, and shows how to **inspect the outputs**.